# Detection-criteria adherence across the sweep

This notebook audits whether serialized detected lightcurves satisfy criteria that can be checked from the stored timestamps and counts. It does not rerun simulations.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('results/detection_criteria_sweep')
QUALITY = ROOT / 'lightcurve_quality.csv'
if not QUALITY.exists():
    raise FileNotFoundError(f'Run lightcurve_quality_analysis.py first: {QUALITY}')
quality = pd.read_csv(QUALITY)
quality.head()

## Measurable adherence checks

The audit uses the detected positive timestamps stored in each result. A source passes the checks below when its stored lightcurve has enough positive points, fits within the configured timespan, and has no positive-detection gap shorter than the configured minimum separation.

In [ ]:
numeric_columns = [
    'n_positive', 'positive_span_days', 'configured_max_timespan_days',
    'min_positive_gap_days', 'criteria_min_detections',
    'criteria_min_time_separation_hours', 'criteria_min_detections_primary',
]
quality[numeric_columns] = quality[numeric_columns].apply(pd.to_numeric, errors='coerce')
quality['min_positive_gap_hours'] = quality['min_positive_gap_days'] * 24.0
quality['passes_min_detections'] = quality['n_positive'] >= quality['criteria_min_detections']
quality['passes_max_timespan'] = quality['positive_span_days'] <= quality['configured_max_timespan_days']
quality['passes_min_time_separation'] = (
    (quality['n_positive'] < 2)
    | (quality['min_positive_gap_hours'] >= quality['criteria_min_time_separation_hours'])
)
quality['passes_measurable_criteria'] = quality[[
    'passes_min_detections', 'passes_max_timespan', 'passes_min_time_separation'
]].all(axis=1)
quality[['passes_min_detections', 'passes_max_timespan', 'passes_min_time_separation', 'passes_measurable_criteria']].mean()

In [ ]:
grid_columns = [
    'criteria_snr_threshold', 'criteria_snr_threshold_secondary',
    'criteria_min_detections', 'criteria_min_detections_primary',
    'configured_max_timespan_days', 'criteria_min_time_separation_hours',
    'criteria_min_fade_rate', 'criteria_min_galactic_lat',
]
adherence_summary = (
    quality.groupby(grid_columns, dropna=False)
    .agg(
        detected_sources=('source_index', 'size'),
        min_detections_pass_rate=('passes_min_detections', 'mean'),
        max_timespan_pass_rate=('passes_max_timespan', 'mean'),
        min_separation_pass_rate=('passes_min_time_separation', 'mean'),
        measurable_criteria_pass_rate=('passes_measurable_criteria', 'mean'),
        median_span_days=('positive_span_days', 'median'),
        median_gap_hours=('min_positive_gap_hours', 'median'),
    )
    .reset_index()
)
adherence_summary.sort_values('measurable_criteria_pass_rate').head(20)

## Violations and diagnostics

Rows below show the worst measurable adherence combinations. The `all_observation_span` check is diagnostic only: non-detections may be stored outside the qualifying positive-detection window.

In [ ]:
violations = quality.loc[~quality['passes_measurable_criteria']].copy()
violation_counts = pd.Series({
    'minimum detections': (~quality['passes_min_detections']).sum(),
    'maximum timespan': (~quality['passes_max_timespan']).sum(),
    'minimum time separation': (~quality['passes_min_time_separation']).sum(),
    'any measurable criterion': (~quality['passes_measurable_criteria']).sum(),
})
print(violation_counts)
violations.sort_values(['positive_span_days', 'min_positive_gap_hours'], ascending=False)[[
    'file', 'source_index', 'n_positive', 'positive_span_days',
    'configured_max_timespan_days', 'min_positive_gap_hours',
    'criteria_min_time_separation_hours', 'criteria_min_detections'
]].head(25)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
adherence_summary.plot.scatter(
    x='configured_max_timespan_days', y='measurable_criteria_pass_rate',
    c='criteria_snr_threshold', colormap='viridis', ax=axes[0],
)
axes[0].set_ylabel('pass rate for measurable criteria')
axes[0].set_xlabel('configured maximum timespan [days]')
quality['positive_span_days'].plot.hist(bins=60, ax=axes[1])
axes[1].set_xlabel('positive-detection span [days]')
axes[1].axvline(14, color='black', linestyle='--', label='14 days')
axes[1].legend()
fig.tight_layout()

## Criteria that require richer serialization

The CSV can report configured SNR thresholds, fade rates, Galactic latitude, and primary-detection requirements, but it cannot prove those tests were applied because the serialized source records do not include per-observation SNR, peak/rise/fade classification, Galactic latitude, or primary/secondary labels. Treat those fields as configuration metadata until the simulator serializes the detector decision details.